# BirdCLEF+ 2026 - Advanced Training (SED + K-Fold)

Medal-worthy approach:
- **SED (Sound Event Detection)** model with attention pooling
- **K-Fold cross-validation** (5 folds) for robust ensembles
- **Focal Loss** to handle extreme class imbalance
- **Mixup + CutMix** augmentation
- **Soundscape fine-tuning** to close the domain gap

**Environment:** Kaggle GPU notebook (T4/P100). Run once per fold.

In [ ]:
import os
import sys
import gc

# ========== CONFIGURATION ==========
FOLD = 0  # Change this for each fold: 0, 1, 2, 3, 4
N_FOLDS = 5
EXPERIMENT = "sed_effb1_focal"  # experiment name for tracking
# ===================================

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    KAGGLE_USER = "stochastix"
    sys.path.insert(0, f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-src")
    DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
    OUTPUT_DIR = "/kaggle/working"
else:
    sys.path.insert(0, "..")
    DATA_DIR = "../data"
    OUTPUT_DIR = "../models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Fold: {FOLD}/{N_FOLDS}, Experiment: {EXPERIMENT}")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from torch.amp import GradScaler, autocast
import yaml
from tqdm.auto import tqdm

from src.dataset import BirdCLEFDataset, SoundscapeDataset, BalancedSampler, load_taxonomy, create_kfold_splits
from src.model import get_model
from src.transforms import get_audio_transforms, get_mixup_fn, get_cutmix_fn
from src.evaluate import evaluate_roc_auc
from src.losses import get_criterion
from src.utils import set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Load config
config_path = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-src/config/sed_efficientnet_b1.yaml" if ON_KAGGLE else "../config/sed_efficientnet_b1.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

# Override paths for environment
config["data"]["train_audio_dir"] = os.path.join(DATA_DIR, "train_audio")
config["data"]["train_csv"] = os.path.join(DATA_DIR, "train.csv")
config["data"]["taxonomy_csv"] = os.path.join(DATA_DIR, "taxonomy.csv")
config["data"]["soundscape_labels_csv"] = os.path.join(DATA_DIR, "train_soundscapes_labels.csv")
config["data"]["train_soundscapes_dir"] = os.path.join(DATA_DIR, "train_soundscapes")

EPOCHS = config["training"]["epochs"]
BATCH_SIZE = config["training"]["batch_size"]
LR = config["training"]["lr"]
SEED = config["data"]["seed"]

set_seed(SEED)
print(f"Config: {EPOCHS} epochs, BS={BATCH_SIZE}, LR={LR}")
print(f"Model: {config['model']['backbone']} ({config['model']['model_type']})")
print(f"Loss: {config['training'].get('loss_type', 'bce')}")

## Data Preparation with K-Fold

In [ ]:
# Load species list
species_list = load_taxonomy(config["data"]["taxonomy_csv"])
print(f"Species: {len(species_list)}")

# Load metadata
train_df = pd.read_csv(config["data"]["train_csv"])
print(f"Total recordings: {len(train_df)}")

# Filter by rating (less aggressive than baseline to keep more data)
min_rating = config["data"].get("min_rating", 0)
if min_rating > 0 and "rating" in train_df.columns:
    train_df = train_df[train_df["rating"] >= min_rating].reset_index(drop=True)
    print(f"After rating filter (>= {min_rating}): {len(train_df)}")

# Create K-fold splits
train_df = create_kfold_splits(train_df, n_folds=N_FOLDS, seed=SEED)
print(f"\nFold distribution:")
print(train_df["fold"].value_counts().sort_index())

# Split for current fold
train_split = train_df[train_df["fold"] != FOLD].reset_index(drop=True)
val_split = train_df[train_df["fold"] == FOLD].reset_index(drop=True)
print(f"\nFold {FOLD}: Train={len(train_split)}, Val={len(val_split)}")

In [ ]:
# Audio transforms (training only)
audio_transforms = get_audio_transforms(config.get("augmentation", {}), train=True)

# Train dataset
train_dataset = BirdCLEFDataset(
    df=train_split,
    audio_dir=config["data"]["train_audio_dir"],
    config=config,
    species_list=species_list,
    audio_transforms=audio_transforms,
    is_train=True,
    secondary_weight=config["data"].get("secondary_label_weight", 0.5),
)

# Optionally add soundscape data
if config["data"].get("use_soundscapes", False):
    sc_labels_path = config["data"]["soundscape_labels_csv"]
    sc_dir = config["data"]["train_soundscapes_dir"]
    if os.path.exists(sc_labels_path) and os.path.exists(sc_dir):
        labels_df = pd.read_csv(sc_labels_path)
        sc_dataset = SoundscapeDataset(
            soundscape_dir=sc_dir,
            config=config,
            species_list=species_list,
            labels_df=labels_df,
            is_test=False,
            augment=True,
            audio_transforms=audio_transforms,
        )
        train_dataset = ConcatDataset([train_dataset, sc_dataset])
        print(f"Added {len(sc_dataset)} soundscape windows")

val_dataset = BirdCLEFDataset(
    df=val_split,
    audio_dir=config["data"]["train_audio_dir"],
    config=config,
    species_list=species_list,
    audio_transforms=None,
    is_train=False,
)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

In [ ]:
# DataLoaders with optional balanced sampling
use_balanced = config["data"].get("balanced_sampling", False)

if use_balanced and isinstance(train_dataset, BirdCLEFDataset):
    sampler = BalancedSampler(train_split, species_list)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=4, pin_memory=True, drop_last=True
    )
    print("Using balanced sampling")
else:
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True, drop_last=True
    )

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Verify shapes
spec, labels = next(iter(train_loader))
print(f"Spectrogram: {spec.shape}, Labels: {labels.shape}")

## Model, Optimizer, Loss

In [ ]:
# Model
model = get_model(config["model"]).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}, Trainable: {trainable_params:,}")

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR,
    weight_decay=config["training"].get("weight_decay", 0.01)
)

# Scheduler
warmup_epochs = config["training"].get("warmup_epochs", 3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - warmup_epochs, eta_min=LR * 0.01
)

# Loss
criterion = get_criterion(config["training"])
print(f"Loss: {criterion.__class__.__name__}")

# Augmentation functions
aug_cfg = config.get("augmentation", {})
mixup_fn = get_mixup_fn(aug_cfg.get("mixup_alpha", 0.4)) if aug_cfg.get("mixup_alpha", 0) > 0 else None
cutmix_fn = get_cutmix_fn(aug_cfg.get("cutmix_alpha", 1.0)) if aug_cfg.get("cutmix_alpha", 0) > 0 else None

# AMP
scaler = GradScaler("cuda") if config["training"].get("amp", True) and device.type == "cuda" else None
grad_clip = config["training"].get("gradient_clip", 0)

## Training Loop

In [ ]:
best_auc = 0.0
history = {"loss": [], "val_auc": [], "lr": []}
model_type = config["model"].get("model_type", "simple")

for epoch in range(1, EPOCHS + 1):
    # === Warmup ===
    if epoch <= warmup_epochs:
        warmup_lr = LR * epoch / warmup_epochs
        for pg in optimizer.param_groups:
            pg["lr"] = warmup_lr

    # === Train ===
    model.train()
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for specs, labels in pbar:
        specs, labels = specs.to(device), labels.to(device)

        # Randomly choose augmentation
        rand = np.random.random()
        if mixup_fn is not None and rand < aug_cfg.get("mixup_prob", 0.5):
            specs, labels, _ = mixup_fn(specs, labels)
        elif cutmix_fn is not None and rand < (aug_cfg.get("mixup_prob", 0.5) + aug_cfg.get("cutmix_prob", 0.3)):
            specs, labels, _ = cutmix_fn(specs, labels)

        optimizer.zero_grad()

        if scaler:
            with autocast("cuda"):
                output = model(specs)
                logits = output["clipwise_logits"] if isinstance(output, dict) else output
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            output = model(specs)
            logits = output["clipwise_logits"] if isinstance(output, dict) else output
            loss = criterion(logits, labels)
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({"loss": f"{running_loss/num_batches:.4f}"})

    avg_loss = running_loss / max(num_batches, 1)

    if epoch > warmup_epochs:
        scheduler.step()

    # === Validate ===
    val_auc = evaluate_roc_auc(model, val_loader, device, species_list, model_type=model_type)

    current_lr = optimizer.param_groups[0]["lr"]
    history["loss"].append(avg_loss)
    history["val_auc"].append(val_auc)
    history["lr"].append(current_lr)

    print(f"Epoch {epoch}: loss={avg_loss:.4f}, val_auc={val_auc:.4f}, lr={current_lr:.6f}")

    if val_auc > best_auc:
        best_auc = val_auc
        save_path = os.path.join(OUTPUT_DIR, f"{EXPERIMENT}_fold{FOLD}_best.pth")
        torch.save({
            "model_state_dict": model.state_dict(),
            "fold": FOLD,
            "epoch": epoch,
            "val_auc": val_auc,
            "config": config,
        }, save_path)
        print(f"  -> Saved best model (AUC: {best_auc:.4f})")

print(f"\nFold {FOLD} complete. Best Val ROC-AUC: {best_auc:.4f}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history["loss"])
axes[0].set_title(f"Training Loss (Fold {FOLD})")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_auc"])
axes[1].axhline(y=best_auc, color='r', linestyle='--', label=f'Best: {best_auc:.4f}')
axes[1].set_title(f"Validation ROC-AUC (Fold {FOLD})")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history["lr"])
axes[2].set_title("Learning Rate")
axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

## Save OOF Predictions (for stacking / threshold tuning)

In [ ]:
from src.evaluate import get_predictions

# Load best model
checkpoint = torch.load(os.path.join(OUTPUT_DIR, f"{EXPERIMENT}_fold{FOLD}_best.pth"), map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

# Get OOF predictions
oof_probs, oof_labels = get_predictions(model, val_loader, device, model_type=model_type)

# Save
np.save(os.path.join(OUTPUT_DIR, f"{EXPERIMENT}_fold{FOLD}_oof_probs.npy"), oof_probs)
np.save(os.path.join(OUTPUT_DIR, f"{EXPERIMENT}_fold{FOLD}_oof_labels.npy"), oof_labels)

print(f"OOF predictions saved: {oof_probs.shape}")
print(f"OOF ROC-AUC: {best_auc:.4f}")

## Per-Species Analysis

In [ ]:
from src.evaluate import compute_roc_auc_per_class

# Binarize labels for evaluation
oof_labels_binary = (oof_labels > 0.5).astype(np.float32)
per_class_auc, macro_auc = compute_roc_auc_per_class(oof_labels_binary, oof_probs, species_list)

# Sort by AUC
sorted_species = sorted(per_class_auc.items(), key=lambda x: x[1])

print(f"\nMacro AUC: {macro_auc:.4f} (from {len(per_class_auc)} species with positives)")
print(f"\nWorst 10 species:")
for species, auc in sorted_species[:10]:
    print(f"  {species}: {auc:.4f}")

print(f"\nBest 10 species:")
for species, auc in sorted_species[-10:]:
    print(f"  {species}: {auc:.4f}")

## Next Steps

After training all 5 folds:
1. Upload all fold weights as a Kaggle dataset: `birdclef-model-weights`
2. Run `05_ensemble_inference.ipynb` which loads all folds and averages predictions
3. Experiment ideas:
   - Try `tf_efficientnet_b2_ns` or `eca_nfnet_l0` backbones
   - Fine-tune on soundscape data only for last 5 epochs
   - Add TTA (time-shift, horizontal flip of spectrograms)
   - Try different loss functions (asymmetric loss)
   - Train with different random seeds for additional diversity